In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import gc
import h5py
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

print("[STEP 0] Setup")

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

DATASET_ROOT = Path("/kaggle/input/skippd")
HDF5_FILE = DATASET_ROOT / "pv_2019.hdf5"

OUTPUT_ROOT = Path("/kaggle/working/skippd_prepared")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

IMG_H, IMG_W, IMG_C = 64, 64, 3
SEQ_LEN = 8
BATCH_SIZE = 16
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TARGET_YEAR = 2019
AUTOTUNE = tf.data.AUTOTUNE 

2026-03-24 14:47:18.516561: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774363638.795232      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774363638.873812      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774363639.514912      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774363639.514958      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774363639.514961      55 computation_placer.cc:177] computation placer alr

[STEP 0] Setup


In [ ]:
print("\n[STEP 1] Loading dataset")

def load_skippd_2019(hdf5_path, target_year=2019):
    with h5py.File(hdf5_path, "r") as f:
        print("Keys:", list(f.keys()))

        images = f["images"][:]

        if "pv_power" in f:
            labels = f["pv_power"][:]
        elif "irradiance" in f:
            labels = f["irradiance"][:]
        else:
            raise KeyError("Label key not found")

        if "datetimes" in f:
            raw_times = f["datetimes"][:]
            if isinstance(raw_times[0], (bytes, np.bytes_)):
                timestamps = pd.to_datetime([t.decode("utf-8") for t in raw_times])
            else:
                timestamps = pd.to_datetime(raw_times)
        elif "timestamps" in f:
            raw_times = f["timestamps"][:]
            timestamps = pd.to_datetime(raw_times, unit="s", utc=True)
        else:
            raise KeyError("Timestamp key not found")

    mask = timestamps.year == target_year
    return images[mask], labels[mask], timestamps[mask]


images_2019, labels_2019, timestamps_2019 = load_skippd_2019(HDF5_FILE, TARGET_YEAR)

print("Loaded samples:", len(images_2019))
print("Image shape:", images_2019.shape)